In [1]:
import os
import json
import cv2
import shutil
from glob import glob
from tqdm import tqdm

REF_YOLO_DIR = 'final_dataset_yoloOnly'
UNITY_IMGS = 'unity/images'
UNITY_JSONS = 'unity/labels'

OUTPUT_BASE = 'lpr_datasets_v2'
OUTPUT_YOLO_ZONE = os.path.join(OUTPUT_BASE, 'yolo_zone')
OUTPUT_LPR_CROPS = os.path.join(OUTPUT_BASE, 'lpr_crops')

for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_YOLO_ZONE, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_YOLO_ZONE, 'labels', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_LPR_CROPS, split), exist_ok=True)

def get_merged_bbox(char_objects):
    x1 = min(c['bbox'][0] for c in char_objects)
    y1 = min(c['bbox'][1] for c in char_objects)
    x2 = max(c['bbox'][2] for c in char_objects)
    y2 = max(c['bbox'][3] for c in char_objects)
    return [x1, y1, x2, y2]

def is_number(text):
    allowed = set("0123456789.-")
    return all(c in allowed for c in text) and len(text) > 0

def process_split(split_name):
    print(f"--- Обробка спліта: {split_name.upper()} ---")
    ref_path = os.path.join(REF_YOLO_DIR, 'labels', split_name)
    files = glob(os.path.join(ref_path, '*.txt'))
    valid_names = set(os.path.basename(f).replace('.txt', '') for f in files)
    print(f"Файлів у спліті: {len(valid_names)}")

    lpr_file_list = [] # train.txt / val.txt

    for name in tqdm(valid_names):
        json_path = os.path.join(UNITY_JSONS, name + '.json')
        img_path_jpg = os.path.join(UNITY_IMGS, name + '.jpg')
        img_path_png = os.path.join(UNITY_IMGS, name + '.png')

        img_path = img_path_jpg if os.path.exists(img_path_jpg) else img_path_png

        if not os.path.exists(json_path) or not os.path.exists(img_path):
            continue

        shutil.copy(img_path, os.path.join(OUTPUT_YOLO_ZONE, 'images', split_name, os.path.basename(img_path)))

        with open(json_path, 'r') as f:
            data = json.load(f)

        img = cv2.imread(img_path)
        h_img, w_img = img.shape[:2]

        yolo_zone_lines = []

        items = data if isinstance(data, list) else [data]
        for item in items:
            chars_pool = item.get('chars', [])
            raw_groups = item.get('data_raw', [])

            cursor = 0
            for group in raw_groups:
                for text_word in group:
                    word_len = len(text_word)
                    current_chars = chars_pool[cursor : cursor + word_len]
                    cursor += word_len

                    if is_number(text_word) and current_chars:
                        bbox = get_merged_bbox(current_chars)

                        bw = bbox[2] - bbox[0]
                        bh = bbox[3] - bbox[1]
                        cx = bbox[0] + bw/2
                        cy = bbox[1] + bh/2

                        yolo_zone_lines.append(f"0 {cx/w_img:.6f} {cy/h_img:.6f} {bw/w_img:.6f} {bh/h_img:.6f}")

                        pad = 2
                        x1 = max(0, int(bbox[0]) - pad)
                        y1 = max(0, int(bbox[1]) - pad)
                        x2 = min(w_img, int(bbox[2]) + pad)
                        y2 = min(h_img, int(bbox[3]) + pad)

                        crop = img[y1:y2, x1:x2]
                        if crop.size == 0: continue

                        unique_id = f"{len(lpr_file_list)}"
                        crop_name = f"{name}_{text_word}_{unique_id}.jpg"
                        crop_save_path = os.path.join(OUTPUT_LPR_CROPS, split_name, crop_name)

                        cv2.imwrite(crop_save_path, crop)

                        lpr_file_list.append(f"{os.path.abspath(crop_save_path)} {text_word}")

        if yolo_zone_lines:
            txt_path = os.path.join(OUTPUT_YOLO_ZONE, 'labels', split_name, name + '.txt')
            with open(txt_path, 'w') as f:
                f.write("\n".join(yolo_zone_lines))

    list_path = os.path.join(OUTPUT_BASE, f'lpr_{split_name}.txt')
    with open(list_path, 'w') as f:
        f.write("\n".join(lpr_file_list))
    print(f"Створено список LPRNet: {list_path}")

process_split('train')
process_split('val')

yaml_content = f"""
path: {os.path.abspath(OUTPUT_YOLO_ZONE)}
train: images/train
val: images/val
names:
  0: number_zone
"""
with open(os.path.join(OUTPUT_BASE, 'yolo_zone_v2.yaml'), 'w') as f:
    f.write(yaml_content)

print("\n✅ Dataset V2 готовий!")

--- Обробка спліта: TRAIN ---
Файлів у спліті: 5590


100%|███████████████████████████████████████████████████████████████████████████████| 5590/5590 [05:43<00:00, 16.29it/s]


Створено список LPRNet: lpr_datasets_v2/lpr_train.txt
--- Обробка спліта: VAL ---
Файлів у спліті: 699


100%|█████████████████████████████████████████████████████████████████████████████████| 699/699 [00:40<00:00, 17.31it/s]

Створено список LPRNet: lpr_datasets_v2/lpr_val.txt

✅ Dataset V2 готовий!
